# Lexical Analysis and Syntax Checking of Fare Calculation Expressions

## Problem Statement

A ride-sharing company requires a compiler front-end module to analyze arithmetic expressions used for fare calculation.

The expression considered is:

fareAmount = (baseFare + distanceKm * ratePerKm) * surgeMultiplier - promoDiscount

The system performs lexical analysis, syntax analysis, parse-tree construction, and error detection.

# 2. Objective

- Identify and classify tokens in the fare expression.
- Perform lexical analysis.
- Check syntax using a Context-Free Grammar.
- Construct a parse tree.
- Detect lexical and syntax errors.
- Implement the compiler front-end using Python.

# 3. Token Definitions

| Token | Example | Type |
|---|---|---|
| fareAmount | fareAmount | Identifier |
| = | = | Assignment Operator |
| ( ) | ( ) | Delimiter |
| + | + | Operator |
| - | - | Operator |
| * | * | Operator |
| / | / | Operator |
| 100 | 100 | Constant |

# 4. Context-Free Grammar

Assignment → ID = Expression

Expression → Term Expression'
Expression' → + Term Expression'
            | - Term Expression'
            | ε

Term → Factor Term'
Term' → * Factor Term'
      | / Factor Term'
      | ε

Factor → ( Expression )
       | ID
       | NUMBER

# 5. Lexical Analyzer

## Aim

To scan the fare calculation expression and identify different tokens such as identifiers, constants, operators, delimiters, and assignment operators.

## Method

The lexer scans the expression character by character. Letters are grouped as identifiers, digits are grouped as constants, and operators and delimiters are classified according to their symbols.

If an unknown symbol is encountered, a lexical error is reported.

## Input

fareAmount = (baseFare + distanceKm * ratePerKm) * surgeMultiplier - promoDiscount

## Expected Output

The program should display each token along with its classification.

In [1]:
def lexer(s):
    tokens = []
    i = 0

    while i < len(s):
        if s[i].isspace():
            i += 1

        elif s[i].isalpha() or s[i] == '_':
            j = i
            while j < len(s) and (s[j].isalnum() or s[j] == '_'):
                j += 1
            tokens.append(("ID", s[i:j]))
            i = j

        elif s[i].isdigit():
            j = i
            while j < len(s) and (s[j].isdigit() or s[j] == '.'):
                j += 1
            tokens.append(("NUMBER", s[i:j]))
            i = j

        elif s[i] in "+-*/":
            tokens.append(("OPERATOR", s[i]))
            i += 1

        elif s[i] == '=':
            tokens.append(("ASSIGN", "="))
            i += 1

        elif s[i] in "()":
            tokens.append(("DELIMITER", s[i]))
            i += 1

        else:
            print("Lexical Error: Invalid symbol", s[i])
            return []

    return tokens


expr = "fareAmount = (baseFare + distanceKm * ratePerKm) * surgeMultiplier - promoDiscount"

tokens = lexer(expr)

print("TOKENS:")
for token in tokens:
    print(token)

TOKENS:
('ID', 'fareAmount')
('ASSIGN', '=')
('DELIMITER', '(')
('ID', 'baseFare')
('OPERATOR', '+')
('ID', 'distanceKm')
('OPERATOR', '*')
('ID', 'ratePerKm')
('DELIMITER', ')')
('OPERATOR', '*')
('ID', 'surgeMultiplier')
('OPERATOR', '-')
('ID', 'promoDiscount')


# 6. Syntax Analyzer

## Aim

To verify whether the sequence of tokens generated by the lexical analyzer follows the defined Context-Free Grammar.

## Method

A recursive-descent parser is used. Separate functions are used for:

- Expression
- Term
- Factor
- Assignment

The parser checks the tokens according to the grammar and reports an error when the expected token is not found.

## Expected Output

Syntax: VALID

In [3]:
class Parser:
    def __init__(self, tokens):
        self.t = tokens
        self.i = 0

    def eat(self, typ):
        if self.i < len(self.t) and self.t[self.i][0] == typ:
            x = self.t[self.i]
            self.i += 1
            return x
        raise Exception("Syntax Error")

    def factor(self):
        if self.i < len(self.t) and self.t[self.i][1] == '(':
            self.eat("DELIMITER")
            x = self.expr()

            if self.i >= len(self.t) or self.t[self.i][1] != ')':
                raise Exception("Syntax Error: Missing )")

            self.eat("DELIMITER")
            return x

        if self.i < len(self.t) and self.t[self.i][0] in ["ID", "NUMBER"]:
            return self.eat(self.t[self.i][0])[1]

        raise Exception("Syntax Error: Expected identifier or number")

    def term(self):
        x = self.factor()

        while self.i < len(self.t) and self.t[self.i][1] in "*/":
            op = self.eat("OPERATOR")[1]
            x = [op, x, self.factor()]

        return x

    def expr(self):
        x = self.term()

        while self.i < len(self.t) and self.t[self.i][1] in "+-":
            op = self.eat("OPERATOR")[1]
            x = [op, x, self.term()]

        return x

    def parse(self):
        self.eat("ID")
        self.eat("ASSIGN")
        tree = self.expr()

        if self.i != len(self.t):
            raise Exception("Syntax Error: Unexpected token")

        return tree


try:
    tree = Parser(tokens).parse()
    print("Syntax: VALID")
except Exception as e:
    print(e)

Syntax: VALID


# 7. Parse Tree Construction

## Aim

To display the hierarchical structure of the valid arithmetic expression.

## Method

The parser stores the operators and operands in a tree-like structure. The root represents the final operation and its children represent the operands and sub-expressions.

The generated parse tree demonstrates the order in which the arithmetic expression is structured according to the CFG.

In [4]:
def show_tree(x, level=0):
    print("  " * level + str(x[0] if isinstance(x, list) else x))

    if isinstance(x, list):
        show_tree(x[1], level + 1)
        show_tree(x[2], level + 1)


print("PARSE TREE:")
show_tree(tree)

PARSE TREE:
-
  *
    +
      baseFare
      *
        distanceKm
        ratePerKm
    surgeMultiplier
  promoDiscount


# 8. Valid Test Cases

The parser is tested with different syntactically correct fare calculation expressions.

The following cases should be accepted:

1. fareAmount = baseFare + distanceKm * ratePerKm
2. fareAmount = (baseFare + distanceKm) * surgeMultiplier
3. fareAmount = baseFare + 100

These test cases verify that the parser accepts identifiers, constants, operators, and parentheses according to the grammar.

In [5]:
tests = [
    "fareAmount = baseFare + distanceKm * ratePerKm",
    "fareAmount = (baseFare + distanceKm) * surgeMultiplier",
    "fareAmount = baseFare + 100"
]

for x in tests:
    try:
        Parser(lexer(x)).parse()
        print("VALID:", x)
    except Exception as e:
        print("INVALID:", x)

VALID: fareAmount = baseFare + distanceKm * ratePerKm
VALID: fareAmount = (baseFare + distanceKm) * surgeMultiplier
VALID: fareAmount = baseFare + 100


# 9. Invalid Test Cases

## Test Case 1 – Invalid Symbol

Input:

fareAmount = baseFare + @rate

The symbol @ is not defined as a valid token. Therefore, the lexical analyzer should report an invalid symbol error.

In [6]:
x = "fareAmount = baseFare + @rate"

lexer(x)

Lexical Error: Invalid symbol @


[]

# 10. Invalid Test Case – Mismatched Parentheses

Input:

fareAmount = (baseFare + distanceKm

The opening parenthesis does not have a corresponding closing parenthesis. The parser should detect the missing closing parenthesis.

In [7]:
x = "fareAmount = (baseFare + distanceKm"

try:
    Parser(lexer(x)).parse()
except Exception as e:
    print(e)

Syntax Error: Missing )


# 11. Invalid Test Case – Missing Operator

Input:

fareAmount = baseFare distanceKm

Two identifiers occur next to each other without an operator. According to the grammar, an operator is required between them.

In [8]:
x = "fareAmount = baseFare distanceKm"

try:
    Parser(lexer(x)).parse()
except Exception as e:
    print(e)

Syntax Error: Unexpected token


# 12. Results and Validation

The developed compiler front-end was tested using valid and invalid fare calculation expressions.

| Test Case | Type | Result |
|---|---|---|
| Original fare expression | Valid | Accepted |
| Simple arithmetic expression | Valid | Accepted |
| Parenthesized expression | Valid | Accepted |
| Invalid @ symbol | Lexical Error | Detected |
| Missing closing parenthesis | Syntax Error | Detected |
| Missing operator | Syntax Error | Detected |

The results show that the system successfully performs lexical analysis, syntax checking, parse-tree construction, and error detection.

# 13. Analysis and Discussion

The custom hand-written lexer and recursive-descent parser were selected because they are simple, transparent, and do not require external libraries.

The lexical analyzer scans the input expression from left to right and classifies tokens. Therefore, its approximate time complexity is O(n), where n is the length of the input.

The recursive-descent parser processes the generated tokens according to the CFG. For the defined grammar, syntax analysis is approximately O(n), where n is the number of tokens.

### Advantages

- Simple and easy to understand.
- No external libraries are required.
- Easy to modify for additional operators.
- Directly represents the CFG through parser functions.
- Detects common lexical and syntax errors.

### Limitations

- Supports only the grammar defined for this project.
- Does not implement semantic analysis.
- Does not evaluate the final fare amount.
- Error messages can be improved by reporting exact positions.

# 14. Conclusion

The project successfully implemented a simple compiler front-end for ride-sharing fare calculation expressions using Python.

The lexical analyzer identified identifiers, constants, operators, delimiters, and assignment operators, while the recursive-descent parser verified the syntax according to the defined Context-Free Grammar.

The system successfully accepted valid expressions and detected errors such as invalid symbols, mismatched parentheses, and missing operators. A parse tree was also generated for valid expressions.

This project demonstrates how compiler design concepts such as lexical analysis, Context-Free Grammars, syntax analysis, parsing, and error detection can be applied to a practical real-world application such as ride-sharing fare calculation.